**Purpose:** `v2_bert_embeddings.ipynb` — part of `02_sentiment/news` (see the stage README).

**Inputs:** `/kaggle/input/news-sentiment/dataset.parquet`

**Outputs:** `results/{combo_key}_learning_curve.png`

**Notes:** paths resolve via `src.config` (run `pip install -e .` from the repo root first).


In [ ]:
from src.seeds import SEED_SENTIMENT_NEWS_V2_BERT_EMBEDDINGS


https://www.kaggle.com/code/hugoverssimo/v2-bertemb?scriptVersionId=295282897

In [ ]:
!pip install torch==2.2.2+cu118 torchvision==0.17.2+cu118 torchaudio==2.2.2+cu118 --index-url https://download.pytorch.org/whl/cu118

!pip install transformers==4.36.2 scikit-learn pandas numpy

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU detected")

# =========================================================
# 5. Sector Descriptions
# =========================================================

# https://www.msci.com/downloads/documents/indexes/gics/GICS+Sector+Definitions+2023.pdf

sector_desc = {
    "Energy": (
        "The Energy Sector comprises companies engaged in exploration & "
        "production, refining & marketing, and storage & transportation of oil "
        "and gas and coal & consumable fuels. It also includes companies that "
        "offer oil & gas equipment and services."
    ),
    "Materials": (
        "The Materials Sector includes companies that manufacture chemicals, "
        "construction materials, forest products, glass, paper and related "
        "packaging products, and metals, minerals and mining companies, "
        "including producers of steel."
    ),
    "Industrials": (
        "The Industrials Sector includes manufacturers and distributors of "
        "capital goods such as aerospace & defense, building products, "
        "electrical equipment and machinery and companies that offer "
        "construction & engineering services. It also includes providers of "
        "commercial & professional services and transportation services."
    ),
    "Consumer Discretionary": (
        "The Consumer Discretionary Sector encompasses those businesses that "
        "tend to be the most sensitive to economic cycles. Its manufacturing "
        "segment includes automobiles & components, household durable goods, "
        "leisure products, and textiles & apparel. The services segment "
        "includes hotels, restaurants, and other leisure facilities, as well as "
        "distributors and retailers of consumer discretionary products."
    ),
    "Consumer Staples": (
        "The Consumer Staples Sector comprises companies whose businesses are "
        "less sensitive to economic cycles. It includes manufacturers and "
        "distributors of food, beverages and tobacco and producers of non-durable "
        "household goods and personal products. It also includes distributors "
        "and retailers of consumer staples products including food & drug "
        "retailing companies."
    ),
    "Health Care": (
        "The Health Care Sector includes health care providers and services, "
        "companies that manufacture and distribute health care equipment and "
        "supplies, health care technology firms, and companies involved in "
        "pharmaceuticals and biotechnology research, development, production, "
        "and marketing."
    ),
    "Financials": (
        "The Financials Sector contains companies engaged in banking, financial "
        "services, consumer finance, capital markets, and insurance activities. "
        "It also includes financial exchanges, data services, and mortgage real "
        "estate investment trusts."
    ),
    "Information Technology": (
        "The Information Technology Sector comprises companies offering "
        "software and technology services, and manufacturers and distributors "
        "of technology hardware and equipment including communications "
        "equipment, computers, peripherals, and semiconductor products."
    ),
    "Communication Services": (
        "The Communication Services Sector includes companies that facilitate "
        "communication and offer related content and information through various "
        "mediums, including telecommunications, media and entertainment, and "
        "interactive services and platforms."
    ),
    "Utilities": (
        "The Utilities Sector comprises utility companies such as electric, "
        "gas, and water utilities, as well as independent power producers, "
        "energy traders, and companies involved in renewable generation."
    ),
    "Real Estate": (
        "The Real Estate Sector contains companies engaged in real estate "
        "development and operations as well as firms offering real estate-"
        "related services, including equity real estate investment trusts."
    ),
}

In [ ]:
# =========================================================
# FINBERT + SECTOR EMBEDDING HYBRID CLASSIFIER (FULL EXP PIPELINE)
# =========================================================

import os, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel, AdamW
import matplotlib.pyplot as plt

# ================= CONFIG =================
SEED = SEED_SENTIMENT_NEWS_V2_BERT_EMBEDDINGS
N_SPLITS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "ProsusAI/finbert"
MAX_LEN = 128
SECTOR_MAX_LEN = 64
os.makedirs("results", exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ================= DATA =================
df_raw = pd.read_parquet("/kaggle/input/news-sentiment/dataset.parquet")

with open("/kaggle/input/news-sentiment/sector_sentiment_quotations.json", "r") as f:
    sector_sentiment_quotations = json.load(f)

gpt_testset_ids = []
for sector in sector_sentiment_quotations:
    for expected_sentiment in sector_sentiment_quotations[sector]:
        for _ in sector_sentiment_quotations[sector][expected_sentiment]:
            for quote in sector_sentiment_quotations[sector][expected_sentiment][_]:
                if quote["label"] == expected_sentiment:
                    gpt_testset_ids.append(quote["gkrecordid"] + " (" + sector + ")")

train_df = df_raw[~df_raw["GKGRECORDID"].isin(gpt_testset_ids)].copy()
min_count = train_df.value_counts(["sector", "majority_sentiment"]).values.min()

df = (
    train_df.groupby(["sector", "majority_sentiment"])
    .apply(lambda x: x.sample(min_count, random_state=SEED_SENTIMENT_NEWS_V2_BERT_EMBEDDINGS))
    .reset_index(drop=True)
)

label2id = {l:i for i,l in enumerate(sorted(df["majority_sentiment"].unique()))}
df["label"] = df["majority_sentiment"].map(label2id)

# ================= SECTOR DESCRIPTIONS =================
sector_desc_df = pd.DataFrame({
    "sector": sector_desc.keys(),
    "sector_description": sector_desc.values()
})

# ================= TOKENIZER =================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ================= PRECOMPUTE SECTOR EMBEDDINGS =================
print("🔹 Computing sector embeddings...")
bert_base = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_base.eval()

sector_embeddings = {}
for _, row in sector_desc_df.iterrows():
    enc = tokenizer(row["sector_description"], truncation=True, padding="max_length",
                    max_length=SECTOR_MAX_LEN, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = bert_base(**enc)
    mask = enc["attention_mask"].unsqueeze(-1)
    emb = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
    sector_embeddings[row["sector"]] = emb.squeeze().cpu()

del bert_base
torch.cuda.empty_cache()

# ================= DATASET =================
class SentDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = tokenizer(row["quotation"], truncation=True, padding="max_length",
                        max_length=MAX_LEN, return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "sector_embed": torch.tensor(sector_embeddings[row["sector"]], dtype=torch.float),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
        }

# ================= MODEL =================
class FinBERTSector(nn.Module):
    def __init__(self, num_labels, proj_dim, dropout, unfrozen):
        super().__init__()
        self.bert = AutoModel.from_pretrained(MODEL_NAME)

        for name, param in self.bert.named_parameters():
            if "encoder.layer." in name:
                layer_num = int(name.split("encoder.layer.")[1].split(".")[0])
                param.requires_grad = layer_num >= unfrozen
            else:
                param.requires_grad = False

        self.sector_proj = nn.Sequential(
            nn.Linear(768, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(768 + proj_dim, num_labels)

    def forward(self, input_ids, attention_mask, sector_embed):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1)
        text_vec = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        sector_vec = self.sector_proj(sector_embed)
        x = torch.cat([text_vec, sector_vec], dim=1)
        x = self.dropout(x)
        return self.classifier(x)

# ================= EVAL =================
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, losses = [], [], []
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    with torch.no_grad():
        for batch in loader:
            for k in batch: batch[k] = batch[k].to(DEVICE)
            logits = model(batch["input_ids"], batch["attention_mask"], batch["sector_embed"])
            loss = loss_fn(logits, batch["label"])
            losses.append(loss.item())
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(batch["label"].cpu().tolist())

    return f1_score(all_labels, all_preds, average="macro"), np.mean(losses)

# ================= HYPERPARAM GRID =================
used_grid = [
    {"unfrozen":6, "bert_lr":1e-6, "head_lr":5e-5, "dropout":0.3, "proj_dim":64, "weight_decay":0.00, "bs":32, "epochs":6},
    {"unfrozen":5, "bert_lr":1e-6, "head_lr":5e-5, "dropout":0.4, "proj_dim":128, "weight_decay":0.00, "bs":32, "epochs":6},
    {"unfrozen":4, "bert_lr":8e-5, "head_lr":1e-4, "dropout":0.3, "proj_dim":64, "weight_decay":0.00, "bs":32, "epochs":7},
]

grid = [
    
]

# ================= TRAINING =================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for hp in grid:

    combo_key = str(hp).replace(" ", "")
    combo_results = []
    learning_curve = {e: {"train": [], "val": []} for e in range(1, hp["epochs"]+1)}

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, df["label"])):

        train_loader = DataLoader(SentDataset(df.iloc[train_idx]), batch_size=hp["bs"], shuffle=True)
        val_loader   = DataLoader(SentDataset(df.iloc[val_idx]), batch_size=hp["bs"])

        model = FinBERTSector(len(label2id), hp["proj_dim"], hp["dropout"], hp["unfrozen"]).to(DEVICE)

        optimizer = AdamW([
            {"params": model.bert.encoder.layer[hp["unfrozen"]:].parameters(), "lr": hp["bert_lr"]},
            {"params": model.sector_proj.parameters(), "lr": hp["head_lr"]},
            {"params": model.classifier.parameters(), "lr": hp["head_lr"]},
        ], weight_decay=hp["weight_decay"])

        loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

        for epoch in range(1, hp["epochs"]+1):
            model.train()
            epoch_train_losses = []

            for batch in train_loader:
                for k in batch: batch[k] = batch[k].to(DEVICE)
                optimizer.zero_grad()
                logits = model(batch["input_ids"], batch["attention_mask"], batch["sector_embed"])
                loss = loss_fn(logits, batch["label"])
                loss.backward()
                optimizer.step()
                epoch_train_losses.append(loss.item())

            train_loss = np.mean(epoch_train_losses)
            val_f1, val_loss = evaluate(model, val_loader)
            learning_curve[epoch]["train"].append(train_loss)
            learning_curve[epoch]["val"].append(val_loss)

        train_f1, _ = evaluate(model, train_loader)
        combo_results.append({"fold": fold, "train_macro_f1": train_f1, "val_macro_f1": val_f1})

    print(f"Finished HP: {hp}")

    lc_stats = {
        e: {"train_mean": float(np.mean(v["train"])),
            "train_std":  float(np.std(v["train"])),
            "val_mean":   float(np.mean(v["val"])),
            "val_std":    float(np.std(v["val"]))}
        for e, v in learning_curve.items()
    }

    with open(f"results/{combo_key}_results.json", "w") as f:
        json.dump({"fold_results": combo_results, "learning_curve": lc_stats}, f, indent=2)

    epochs = sorted(lc_stats.keys())
    train_means = [lc_stats[e]["train_mean"] for e in epochs]
    val_means   = [lc_stats[e]["val_mean"] for e in epochs]

    plt.figure()
    plt.plot(epochs, train_means, marker="o", label="Train Loss")
    plt.plot(epochs, val_means, marker="o", label="Validation Loss")
    plt.title(f"Learning Curve\n{combo_key}")
    plt.legend(); plt.grid(True)
    plt.savefig(f"results/{combo_key}_learning_curve.png")
    plt.close()

print("\n🚀 All experiments complete.")
